In [1]:
%pip install opencv-python ultralytics

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
from ultralytics import YOLO

# 1. Load the pre-trained models
# Update the paths to point directly to the .pt files shown in your screenshot
try:
    localization_model = YOLO("/Users/abishekkhadka/Desktop/Bachelor-s-Note/6th Sem/Automated-Two-Wheeler-Violation-Detection-and-E-Challan-System/ML/Code/number_plate/license_plate_detector.pt")
    plate_detector = YOLO("/Users/abishekkhadka/Desktop/Bachelor-s-Note/6th Sem/Automated-Two-Wheeler-Violation-Detection-and-E-Challan-System/ML/Code/number_plate/plate_localization.pt")
except Exception as e:
    print(f"Error loading models: {e}")
    exit()

# 2. Define the path to your video file
# Update this with the full name of the video file in your directory
VIDEO_PATH = "/Users/abishekkhadka/Desktop/Bachelor-s-Note/6th Sem/Automated-Two-Wheeler-Violation-Detection-and-E-Challan-System/ML/Code/number_plate/YTDown_YouTube_Indian-Traffic-Footage-Pixels_Media_4L_VloEggeo_001_1080p.mp4" 

# 3. Initialize OpenCV video capture
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

# Process the video frame by frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # End of video

    # 4. Run inference with the first model (e.g., detecting the general plate area)
    # Using predict() on a single frame
    results_detector = plate_detector.predict(source=frame, conf=0.5, verbose=False)

    # 5. Extract bounding boxes from the first model
    for result in results_detector:
        boxes = result.boxes
        for box in boxes:
            # Extract coordinates (x1, y1, x2, y2)
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Crop the detected region from the original frame
            cropped_region = frame[y1:y2, x1:x2]

            # 6. Run the second model on the cropped region
            # Ensure the crop is valid before running inference
            if cropped_region.size > 0:
                # Run the localization model on the crop
                results_localization = localization_model.predict(source=cropped_region, conf=0.5, verbose=False)

                # Visualize the results of the second model on the crop
                for loc_result in results_localization:
                    # Draw bounding boxes from the second model ON the cropped region
                    annotated_crop = loc_result.plot()
                    
                    # Optionally, paste the annotated crop back into the original frame
                    # This requires ensuring the dimensions match, so displaying it separately might be easier for debugging
                    cv2.imshow("Localized Plate Crop", annotated_crop)
                    
            # Draw the initial detection box on the main frame for context
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)


    # Display the main annotated frame
    cv2.imshow("Main Video Inference", frame)

    # Press 'q' to exit early
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Clean up resources
cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 